In [ ]:
import spacetrack.operators as op
from spacetrack import SpaceTrackClient
import datetime as dt
import re
import json
import numpy as np
from skyfield.api import load
from functions.functions import compute_reflection
from mpl_toolkits.basemap import Basemap
import matplotlib.pyplot as plt
import pymap3d
import os
from dotenv import load_dotenv

import satellite_parameter_utils as utils



##################################################################
############################# INPUTS #############################
##################################################################

#CYGNSS ID, Start time, End time
#cyg01,2018-08-22 14:26:59,2018-08-22 14:27:59

#PRN Code
#20
#14 -> the satellite that had PRN 14 in 2018 is not active nowadays
#10
#32
# CYGNSS ID name
cygnss_satellite_name = "cygfm01"  # Change this to the satellite name you want
##################################################################

# Retrieve access to TLEs to SpaceTrack
dotenv_path = os.path.join(os.path.dirname(os.getcwd()), '.env')
load_dotenv(dotenv_path)

spacetrack_username = os.getenv("SPACETRACK_ID")
spacetrack_password = os.getenv("SPACETRACK_PASS")
print(f"Space-Track API connected with user: {spacetrack_username}")
# Let's keep it private it....
st = SpaceTrackClient(identity=spacetrack_username, password=spacetrack_password)

# Define the PRN codes and the time range
PRN_vec = [20, 10, 32, 14] # From the PlotIFTracks
date_start = dt.datetime(2018, 8, 22, 14, 26, 59)
date_end = dt.datetime(2018, 8, 22, 14, 27, 59)


# Read the GPS (updated) TLE file to relate PRN to NORAD ID
# To be updated when we have all PRN values for non active GPS satellites 
input_file = "../TLE_EPHEMIRIS_files/GPS_operational.tle"# Extract PRN Code and NORAD ID
with open(input_file, "r") as file:
    tle_data = file.read()



# prn_norad_in_tle_file_pattern = re.compile(r"GPS .+\(PRN (\d+)\)\s+1 (\d+)U")
# prn_norad_tuples = prn_norad_in_tle_file_pattern.findall(tle_data)
prn_norad_tuples = []
for prn in PRN_vec:
    # PRN needs to be on the format GXX where XX is the PRN number, the number needs to be 2 digits.
    prn = f"G{prn:02d}"
    #print(prn)

    prn_mapping_data = utils.load_satellite_parameter_mapping(utils.JSON_MAPPING_FILENAME)
    #print(prn_mapping_data[0])
    norad_id = utils.prn_to_norad_id(prn_mapping_data, prn, (date_start, date_end))
    if len(norad_id) > 1:
        raise ValueError(f"Multiple NORAD IDs found for PRN {prn} in the given date range. \
                         This is unexpected behaviour.")
    if len(norad_id) == 0:
        norad_id = utils.prn_to_norad_id(prn_mapping_data, prn)
        #print(norad_id)
        raise ValueError(f"No NORAD ID found for PRN {prn} in the given date range. \
                         This is unexpected behaviour.")
    prn_norad_tuples.append((int(prn[1:]), norad_id[0]))
    
# One day search for the TLEs on spacetrack
date_search_start = date_start.date()
date_search_end = date_start.date() + dt.timedelta(days=1)
date_search_range = op.inclusive_range(date_search_start, date_search_end)
##################################################################
########## Create GPS TLE file for all satellites (PRN) ##########
##################################################################
# Empty the file before writing new data
with open("../TLE_EPHEMIRIS_files/ALL_GPS_TLE.txt", "w") as file:
    file.write("")  # Clears the file

# Now open in append mode to add new data
with open("../TLE_EPHEMIRIS_files/ALL_GPS_TLE.txt", "a") as file:
    for PNR in PRN_vec: 

        match = None
        for earth_visualization_map in prn_norad_tuples:
            if int(earth_visualization_map[0]) == PNR:
                match = earth_visualization_map
                break

        if match:
                
            # NORAD ID
            norad_cat_id = match[1]

            #lines = list(st.tle_publish(iter_lines=True, norad_cat_id=norad_cat_id, publish_epoch=drange, orderby='PUBLISH_EPOCH', format='tle'))
            lines = list(st.tle(iter_lines=True, norad_cat_id=norad_cat_id, epoch=date_search_range, orderby="EPOCH", format="tle"))
            # Access the last two elements (most recent time-wise)
            if lines:
                last_elements = lines[-2:]
                print(last_elements)

                # Write the NORAD ID as a header and the TLE data to the file (no extra spaces)
                file.write(f"NORAD ID: {norad_cat_id}\n")  # Header without extra newlines
                for line in last_elements:
                    file.write(line.strip() + "\n")  # Strip leading/trailing spaces and ensure new lines
            else:
                print(f"No data found for PRN {PNR}.")



##################################################################
#################### Create CYGNSS TLE file ######################
##################################################################
cygnss_satellite_tle_data_json = st.satcat(satname=cygnss_satellite_name, format='json')
# Empty the file before writing new data
with open("../TLE_EPHEMIRIS_files/cygnss_TLE.txt", "w") as file:
    file.write("")  # Clears the file

# Print the results
if cygnss_satellite_tle_data_json:
    # Parse JSON
    cygnss_satellite_data = json.loads(cygnss_satellite_tle_data_json)
    norad_cat_id_cy = cygnss_satellite_data[0].get("NORAD_CAT_ID")
    lines = list(st.tle(iter_lines=True, norad_cat_id=norad_cat_id_cy, epoch=date_search_range, orderby="EPOCH", format="tle"))
    # Access the last two elements (most recent time-wise)
    if lines:
        last_elements = lines[-2:]
        print(last_elements)
    # Now open in append mode to add new data
    with open("../TLE_EPHEMIRIS_files/cygnss_TLE.txt", "a") as file:

            # Write the NORAD ID as a header and the TLE data to the file (no extra spaces)
            file.write(f"NORAD ID: {norad_cat_id_cy}\n")  # Header without extra newlines
            for line in last_elements:
                file.write(line.strip() + "\n")  # Strip leading/trailing spaces and ensure new lines

else:
    print("Satellite not found.")


User ID: sindre.a.a.nordtveit@ntnu.no
['1 26360U 00025A   18234.74896528 -.00000052 +00000-0 +00000-0 0  9999', '2 26360 053.2195 136.8907 0039654 120.7902 149.7268 02.00563248133998']
['1 41019U 15062A   18234.91080196 -.00000049  00000-0  00000+0 0  9994', '2 41019  55.0899 143.1293 0038013 203.7564 156.0918  2.00564521 20566']
['1 41328U 16007A   18234.99893691  .00000010  00000-0  00000+0 0  9994', '2 41328  54.8358 202.8338 0024422 219.4670 140.3284  2.00560666 18622']
['1 26605U 00071A   18234.74422986 +.00000020 +00000-0 +00000-0 0  9999', '2 26605 055.0577 207.7216 0102891 249.1080 262.6114 02.00569930130279']
['1 41887U 16078D   18234.39436576  .00000266  00000-0  16005-4 0  9991', '2 41887  34.9523 229.8814 0016957 231.1427 128.7771 15.13997538 93228']


In [ ]:
# Load the TLE file and create the satellites objects
gps_tle_file_path = "../TLE_EPHEMIRIS_files/ALL_GPS_TLE.txt"
sat_tle_file_path = "../TLE_EPHEMIRIS_files/cygnss_TLE.txt"
satellites_LEO_sk = load.tle_file(sat_tle_file_path)
satellites_GPS_sk = load.tle_file(gps_tle_file_path)


# Plot with basemap for good visualization of Earth surface
plt.figure(figsize=(10, 10))
earth_visualization_map = Basemap(projection='cyl',llcrnrlat=-10,urcrnrlat=60,
            llcrnrlon=10,urcrnrlon=130, resolution='i')

earth_visualization_map.drawcoastlines()
# Define latitude and longitude intervals suitable for zoomed-in view -> only for this case. Make it general
parallels = range(-10, 61, 10)   # Latitude lines every 10 degrees
meridians = range(10, 131, 10)  # Longitude lines every 10 degrees

# Draw latitude and longitude grid lines with labels
earth_visualization_map.drawparallels(parallels, labels=[1, 0, 0, 0], fontsize=10)  # Latitude labels on the left
earth_visualization_map.drawmeridians(meridians, labels=[0, 0, 0, 1], fontsize=10)  # Longitude labels on the bottom

# Generate Seconds time vector to add it to the time object. 
step = 1 # Seconds between each point
seconds_array = np.arange(0, (date_end - date_start).total_seconds(), step)
# Load timescale and generate Skyfield time series
start_time_dt = date_start 
ts = load.timescale()
times_1 = ts.utc(
    start_time_dt.year,
    start_time_dt.month,
    start_time_dt.day,
    start_time_dt.hour,
    start_time_dt.minute,
    start_time_dt.second + seconds_array
)

# Need to create another datime obect, times_2, to run the latitude, longitude conversion
times_2 = [time.utc_datetime() for time in times_1]  # Call the method on each Time object


satellite_leo = satellites_LEO_sk[0]
position2 = np.array([satellite_leo.at(t).position.m       for t in times_1])
#velocity2 = np.array([satellite_leo.at(t).velocity.m_per_s for t in times_1])
position2 = np.transpose(position2)
color_vector = ['r', 'g', 'b', 'y', 'm', 'c', 'k']

def does_track_cross_from_land_to_sea(basemap, track: np.array) -> bool:
    previous_point = None
    for evaluation_point in track:
        if not previous_point: # Handle the first point
            if Basemap.is_land(basemap, evaluation_point[0], evaluation_point[1]):
                previous_point = "land"
            else:
                previous_point = "sea"
            continue
        if Basemap.is_land(basemap, evaluation_point[0], evaluation_point[1]):
            if previous_point == "sea":
                return True
            previous_point = "land"
        else:
            if previous_point == "land":
                return True
            previous_point = "sea"
    return False

# Iterate along different GPS satellites (and PRN codes)
for satellite_gps, prn, color in zip(satellites_GPS_sk, PRN_vec, color_vector):
    # Extract position and velocity vectors
    position1 = np.array([satellite_gps.at(t).position.m for t in times_1])
    position1 = np.transpose(position1)

    # Compute reflection
    specular_point_vec, theta_vec, latlon_ref = compute_reflection(position1, position2, 0, 80, times_2)
    
    # Plot reflection track
    earth_visualization_map.plot(latlon_ref[:, 1], latlon_ref[:, 0], linewidth=2, color=color, label=f"Reflection track {prn}",linestyle='dashed')
    #print(f"Reflection track coordinates {latlon_ref[:,1], latlon_ref[:,0]}")
    # Check if the track crosses from land to sea or vice versa
    if does_track_cross_from_land_to_sea(earth_visualization_map, latlon_ref):
        print(f"Reflection track for PRN {prn} crosses from land to sea or vice versa.")
    # else:
    #     print(f"Reflection track for PRN {prn} does not cross from land to sea or vice versa.")

    # Convert ECI to geodetic coordinates
    lat, lon, alt = pymap3d.eci2geodetic(position1[0, :], position1[1, :], position1[2, :], times_2)
    
    # Plot GPS satellite track
    earth_visualization_map.plot(lon, lat, linewidth=4, color=color, label=f"GPS Satellite {prn}")

lat, lon, alt = pymap3d.eci2geodetic(position2[0, :], position2[1, :], position2[2, :], times_2)
earth_visualization_map.plot(lon, lat, linewidth=2, color='k', label = cygnss_satellite_name)
# add ticks on x, y axis
plt.xticks(fontsize=12)
plt.legend(loc='lower left') # Place in the bottom left location
plt.show()
